# MNIST Simple Neural Network Example

The MNIST Dataset is a set of images, 28 by 28 pixels. These images contain the numbers 0 - 9 (10 numbers) and the associated label.
We will design a model, such that it is rewarded for mapping image data as input, to a number from 0-9. We will compare the outputs from the model to the known label as part of our error function.

## Imports

Unfortunately these labs do not have torchvision installed, so we cant run this section.

In [1]:
# Neural Networks:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from tqdm import tqdm



# Metrics:
from sklearn.metrics import confusion_matrix
import seaborn as sns
from sklearn.metrics import confusion_matrix
import seaborn as sns

## Load Datasets

In [2]:
# Root File Location of the Data:
file_loc = "C:\\Users\\igriffit\\OneDrive - University of South Wales (1)\\Data\\MNIST Data"


# Training Data Location:
load_train = f'{file_loc}\\MNIST_train.pt'
load_train_labels = f'{file_loc}\\MNIST_train_labels.pt'

# Testing Data Location:
load_test = f'{file_loc}\\MNIST_test.pt'
load_test_labels = f'{file_loc}\\MNIST_test_labels.pt'



# Load Training and Testing Data:
# Note: The "to" function, can also be used to change to float.

train_images = torch.load(load_train)
test_images = torch.load(load_test)
train_labels = torch.load(load_train_labels)
test_labels = torch.load(load_test_labels)

## View Datasets

Yoy might be confusted why the dataset is 1, 28, 28? Why the 1? Well when we do image networks, like a CNN we will see later, PyTorch likes to know how many channels ther are. These are black and white images, so are one channel. Coloured images are stored as RGB (Red, Green, Blue) and so they are 3-channel images!

In [3]:
train_images.shape, test_images.shape

(torch.Size([60000, 1, 28, 28]), torch.Size([10000, 1, 28, 28]))

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
sample_num = 400
print(f'The Corrosponding Label is: {train_labels[sample_num]}')
plt.imshow(train_images[sample_num][0], cmap='gray')
plt.show()

## Flatten Data

In [6]:
train_images.shape[3]

28

In [7]:
train_images = train_images.view(train_images.shape[0], train_images.shape[2]*train_images.shape[3])
test_images = test_images.view(test_images.shape[0], test_images.shape[2]*test_images.shape[3])

Our data images are float32, this is the format PyTorch likes for input data. For labelled data, PyTorch prefers the long data type.

In [8]:
print(train_images.shape, test_images.shape)
print(train_images.type(), test_images.type())
print(train_labels.type(), test_labels.type())

torch.Size([60000, 784]) torch.Size([10000, 784])
torch.FloatTensor torch.FloatTensor
torch.LongTensor torch.LongTensor


## Simple Neural Network

Notice that the class structure is identical to our logistic regression model, previously discussed. However, just far more complex in the number of layerys, and neurons utilised.
The design of this network is very flexible, and there isn't an exact science on what network structure will yield the best results.

Here are some of general guidelines:

- **The Input is Fixed**: The **Input** has to match the number of neurons that are present within a single sample of your data. In our case, this is 784. 
- **The Output is Fixed**: The **Output** has to match the number of classes you are looking to predict, in this instance is 10 neurons. 

- **Bottlenecks are generally bad**: Think of Neurons as storing information. If we have the following network structure (1000, 5, 10): We start with 1000 neurons as input (1000 bits of information), then we compress that down to only 5 key bits of information. Then we want to use those 5 bits of information to predict 10 classes. There's a bottle neck in the middle, where we compress the information significantly before going to our 10 classes of predictions. (1000 -> 100 -> 25 -> 10) is a much more reasonable compression in information. 

- **Deep Networks are Complex Networks**: As you add more and more layers, the model over time is able to compress the information into a better machine encoded task, generally when you have complex problems, adding more layers to your networks will help the model encode these complex features in a way it understands. With that said, the computation time of your model goes up drastically, so complexity should only be added for reasonably complex problems.

Notice our class strucutre is identical to our linear class model, and logistic class model.

What is the ReLu activation function? Well it's a popular activation function for hidden layers

### Model 1: A Standard Model


Here is a model that follows the general good pracices i've mentioned above. This model avoids any bottlenecks.

In [9]:
# Create a different style network. 
class Simple_NN1(nn.Module):
    
    def __init__(self, n_features):
        super().__init__()
        self.fc1 = nn.Linear(n_features, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 16)
        self.fc4 = nn.Linear(16, 10)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = F.log_softmax(self.fc4(x))

        return x

In [10]:
net = Simple_NN1(n_features=784)
print(net)

Simple_NN1(
  (fc1): Linear(in_features=784, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=32, bias=True)
  (fc3): Linear(in_features=32, out_features=16, bias=True)
  (fc4): Linear(in_features=16, out_features=10, bias=True)
)


In [11]:
def train(model, images, labels, epochs, batch_size):
    loss_function = nn.CrossEntropyLoss()
    optimizer =  torch.optim.SGD(model.parameters(), lr=0.01)
    
    # Training Loop:
    for epoch in range(epochs):
        for i in tqdm(range(0, len(images), batch_size)):
            # Batch the Data:
            batch_data = images[i:i+batch_size]
            batch_labels =labels[i:i+batch_size]

            # 1) Calculate the Output of Model:
            y_predicted = model.forward(batch_data)

            # 2) Calculate the Error:
            loss = loss_function(y_predicted, batch_labels)

            # 3) Calculate the Gradients (Note: We'll ensure that the gradients are empty first, before we calculate it):
            model.zero_grad() # Reset Gradients
            loss.backward() # Calculate the Gradients

            # 4) Update the Gradients:
            optimizer.step()

        print(f"Iteration {epoch} | Loss: {loss}")






In [14]:
train(net, train_images, train_labels, epochs=10, batch_size=100)

  0%|          | 0/600 [00:00<?, ?it/s]C:\Users\igriffit\AppData\Local\Temp\ipykernel_25272\859838194.py:15: UserWarning: Implicit dimension choice for log_softmax has been deprecated. Change the call to include dim=X as an argument.
  x = F.log_softmax(self.fc4(x))
100%|██████████| 600/600 [00:01<00:00, 576.52it/s]


Iteration 0 | Loss: 0.16301237046718597


100%|██████████| 600/600 [00:01<00:00, 590.37it/s]


Iteration 1 | Loss: 0.15050837397575378


100%|██████████| 600/600 [00:01<00:00, 592.60it/s]


...

Iteration 7 | Loss: 0.10069180279970169


100%|██████████| 600/600 [00:01<00:00, 599.59it/s]


Iteration 8 | Loss: 0.0949193462729454


100%|██████████| 600/600 [00:00<00:00, 708.82it/s]

Iteration 9 | Loss: 0.09007429331541061


In [15]:
def test(model, images, labels):
    model.to('cpu') # Model on CPU for testing.

    with torch.no_grad():
        y_predicted = model.forward(images)

    print(f"Data Examples: {y_predicted[0]} \n\n{labels[0]}")

    predicted_classes = torch.argmax(y_predicted, dim = 1)
    print(predicted_classes[0:5])
    print(labels[0:5])


    # Accuracy:
    correct = 0
    total = 0

    for i in range(len(predicted_classes)):
        if predicted_classes[i] == labels[i]:
            correct += 1
        total += 1

    print(f"Accuracy: {round(correct/total, 3)}")

    # Confusion Matrix:


    conf_matrix = confusion_matrix(labels, predicted_classes)
    print(conf_matrix)
    




In [16]:
test(net, test_images, test_labels)

Data Examples: tensor([-1.8517e+01, -6.8356e+00, -7.2342e+00, -8.1311e+00, -1.4809e+01,
        -1.6026e+01, -1.4763e+01, -2.1707e-03, -1.2825e+01, -9.5102e+00]) 

7
tensor([7, 2, 1, 0, 4])
tensor([7, 2, 1, 0, 4])
Accuracy: 0.949
[[ 964    0    0    2    0    3    7    2    1    1]
 [   0 1116    2    2    0    1    2    1   11    0]
 [  10    3  975    7    4    0   11   10   11    1]
 [   2    1   16  952    0    9    0   10   16    4]
 [   2    1    3    0  924    0   14    1    3   34]
 [   9    3    0   11    4  820   14    1   16   14]
 [   8    3    3    0    9   11  921    0    3    0]
 [   1    9   19    4    5    0    1  969    0   20]
 [   4    3    7   12    7   17    7    5  897   15]
 [  10    6    0    3   18    8    0   12    4  948]]


C:\Users\igriffit\AppData\Local\Temp\ipykernel_25272\859838194.py:15: UserWarning: Implicit dimension choice for log_softmax has been deprecated. Change the call to include dim=X as an argument.
  x = F.log_softmax(self.fc4(x))


### Model 2: The Complex Model

This model, is has multiple layers, with a extremely large number of neurons. More than are even provided via the input. This may not necessarily yield incorrect results, but we will see the issues with this model

In [17]:
class Simple_NN2(nn.Module):
    
    def __init__(self, n_features):
        super().__init__()
        self.fc1 = nn.Linear(n_features, 100) # w1x1 + w2x2 +w3x3 + .....784 network parameters
        self.fc2 = nn.Linear(100, 1000)
        self.fc3 = nn.Linear(1000, 5000)
        self.fc4 = nn.Linear(5000, 10000)
        self.fc5 = nn.Linear(10000, 10000)
        self.fc6 = nn.Linear(10000, 10)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = F.relu(self.fc4(x))
        x = F.relu(self.fc5(x))
        x = F.log_softmax(self.fc6(x), dim = 1)
        #x = self.fc4(x)
        return x

In [18]:
net = Simple_NN2(n_features=784)

train(net, train_images, train_labels, epochs=10, batch_size=100)
test(net, test_images, test_labels)

 11%|█▏        | 68/600 [00:32<04:16,  2.08it/s]


KeyboardInterrupt: 

### Model 3: The Poor Model

Here will create a poor model, it has too many layers for the problem, and has massive bottlenecks!

In [19]:
class Simple_NN3(nn.Module):
    
    def __init__(self, n_features):
        super().__init__()
        self.fc1 = nn.Linear(n_features, 1) # w1x1 + w2x2 +w3x3 + .....784 network parameters
        self.fc2 = nn.Linear(1, 1)
        self.fc3 = nn.Linear(1, 1)
        self.fc4 = nn.Linear(1, 1)
        self.fc5 = nn.Linear(1, 1)
        self.fc6 = nn.Linear(1, 10)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = F.relu(self.fc4(x))
        x = F.relu(self.fc5(x))
        x = F.log_softmax(self.fc6(x), dim = 1)
        #x = self.fc4(x)
        return x

In [20]:
net = Simple_NN3(n_features=784)

train(net, train_images, train_labels, epochs=10, batch_size=100)
test(net, test_images, test_labels)

100%|██████████| 600/600 [00:00<00:00, 867.42it/s]


Iteration 0 | Loss: 2.361941337585449


100%|██████████| 600/600 [00:00<00:00, 684.08it/s]


Iteration 1 | Loss: 2.3187713623046875


100%|██████████| 600/600 [00:00<00:00, 772.85it/s]


...

Iteration 7 | Loss: 2.3014097213745117


100%|██████████| 600/600 [00:00<00:00, 818.68it/s]


Iteration 8 | Loss: 2.301377058029175


100%|██████████| 600/600 [00:00<00:00, 786.37it/s]


Iteration 9 | Loss: 2.301362991333008
Data Examples: tensor([-2.3118, -2.1879, -2.3082, -2.2866, -2.3309, -2.3997, -2.3072, -2.2627,
        -2.3223, -2.3217]) 

7
tensor([1, 1, 1, 1, 1])
tensor([7, 2, 1, 0, 4])
Accuracy: 0.114
[[   0  980    0    0    0    0    0    0    0    0]
 [   0 1135    0    0    0    0    0    0    0    0]
 [   0 1032    0    0    0    0    0    0    0    0]
 [   0 1010    0    0    0    0    0    0    0    0]
 [   0  982    0    0    0    0    0    0    0    0]
 [   0  892    0    0    0    0    0    0    0    0]
 [   0  958    0    0    0    0    0    0    0    0]
 [   0 1028    0    0    0    0    0    0    0    0]
 [   0  974    0    0    0    0    0    0    0    0]
 [   0 1009    0    0    0    0    0    0    0    0]]


### Model 4: The Simple Model.

Out Application, is actually suprisingly simple. We are processing images that are only 28x28 pixels. With a number written simply in the middle and nothing else.
A 4k image has 3840x2160 pixels, and depending on the photo could contain multiple things, a bus, street lights, and much more.

Turns out, we don't need a complex model for this problem.

In [21]:
class Simple_NN4(nn.Module):
    
    def __init__(self, n_features):
        super().__init__()
        self.fc1 = nn.Linear(n_features, 128) # w1x1 + w2x2 +w3x3 + .....784 network parameters
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.log_softmax(self.fc2(x), dim = 1)
        #x = self.fc4(x)
        return x

In [22]:
net = Simple_NN4(n_features=784)

train(net, train_images, train_labels, epochs=10, batch_size=100)
test(net, test_images, test_labels)

100%|██████████| 600/600 [00:00<00:00, 722.51it/s]


Iteration 0 | Loss: 0.8645780086517334


100%|██████████| 600/600 [00:00<00:00, 679.24it/s]


Iteration 1 | Loss: 0.48964041471481323


100%|██████████| 600/600 [00:00<00:00, 796.79it/s]


...

Iteration 7 | Loss: 0.22740516066551208


100%|██████████| 600/600 [00:00<00:00, 685.86it/s]


Iteration 8 | Loss: 0.21606913208961487


100%|██████████| 600/600 [00:00<00:00, 637.72it/s]

Iteration 9 | Loss: 0.20639407634735107
Data Examples: tensor([-9.0670e+00, -1.4903e+01, -8.3130e+00, -6.5977e+00, -1.2285e+01,
        -1.0280e+01, -1.5898e+01, -3.1203e-03, -9.8765e+00, -6.6450e+00]) 

7
tensor([7, 2, 1, 0, 4])
tensor([7, 2, 1, 0, 4])
Accuracy: 0.92
[[ 959    0    3    2    0    3    9    1    3    0]
 [   0 1108    2    2    0    2    4    2   15    0]
 [  10    5  922   14   15    2   11   14   34    5]
 [   3    1   23  913    0   28    2   14   16   10]
 [   1    2    6    0  911    0   12    2    7   41]
 [  11    3    4   35    9  777   17    5   25    6]
 [  13    3    4    0   14   12  907    1    4    0]
 [   3    9   27    6    7    0    0  946    2   28]
 [   8    8    6   23   11   21   14   12  856   15]
 [  10    7    3   11   40    9    0   20    6  903]]


## GPU Training

This is why we're using PyTorch, to use the GPU! The below code allows PyTorch to dynamically change between GPU and CPU depending on whether the user has one.

In [23]:
if torch.cuda.is_available():
    device = torch.device("cuda:0")
    print("running on the GPU")
else:
    device = torch.device("cpu")
    print("running on the CPU")

running on the GPU


### GPU Training Loop

In [24]:
def train(model, images, labels, epochs, batch_size, device):
    # Ensure Model is Loaded onto the GPU:
    print(f"Device = {device}")
    model = model.to(device)


    loss_function = nn.CrossEntropyLoss()
    optimizer =  torch.optim.SGD(model.parameters(), lr=0.01)
    
    # Training Loop:
    for epoch in range(epochs):
        for i in tqdm(range(0, len(images), batch_size)):
            # Batch the Data:
            batch_data = images[i:i+batch_size].to(device)
            batch_labels = labels[i:i+batch_size].to(device)

            # 1) Calculate the Output of Model:
            y_predicted = model.forward(batch_data)

            # 2) Calculate the Error:
            loss = loss_function(y_predicted, batch_labels)

            # 3) Calculate the Gradients (Note: We'll ensure that the gradients are empty first, before we calculate it):
            model.zero_grad() # Reset Gradients
            loss.backward() # Calculate the Gradients

            # 4) Update the Gradients:
            optimizer.step()

        print(f"Iteration {epoch} | Loss: {loss}")


In [25]:
net = Simple_NN1(n_features=784)
train(net, train_images, train_labels, epochs=10, batch_size=100, device=device)
test(net, test_images, test_labels)

Device = cuda:0


  0%|          | 0/600 [00:00<?, ?it/s]C:\Users\igriffit\AppData\Local\Temp\ipykernel_25272\859838194.py:15: UserWarning: Implicit dimension choice for log_softmax has been deprecated. Change the call to include dim=X as an argument.
  x = F.log_softmax(self.fc4(x))
100%|██████████| 600/600 [00:07<00:00, 84.20it/s] 


Iteration 0 | Loss: 2.2441751956939697


100%|██████████| 600/600 [00:01<00:00, 467.79it/s]


Iteration 1 | Loss: 1.6932002305984497


...

Iteration 7 | Loss: 0.27463001012802124


100%|██████████| 600/600 [00:01<00:00, 448.56it/s]


Iteration 8 | Loss: 0.2417343258857727


100%|██████████| 600/600 [00:01<00:00, 454.89it/s]


Iteration 9 | Loss: 0.21552903950214386
Data Examples: tensor([-9.4184e+00, -9.3184e+00, -7.8035e+00, -5.1894e+00, -1.7575e+01,
        -1.0629e+01, -2.1791e+01, -1.0505e-02, -7.7706e+00, -5.5598e+00]) 

7
tensor([7, 2, 1, 0, 4])
tensor([7, 2, 1, 0, 4])
Accuracy: 0.914
[[ 955    0    6    0    0    7    3    3    4    2]
 [   0 1111    4    3    0    1    2    3   11    0]
 [  11   10  946   11    7    1   11    8   20    7]
 [   3    0   34  908    0   19    0   19   20    7]
 [   3    4    3    0  876    0   12    1   11   72]
 [  20    3    8   47    9  742   13    9   31   10]
 [  16    3   10    0   13   19  890    0    5    2]
 [   3   17   18    2    2    0    0  948    1   37]
 [   5   10    9   20   11   30   14   10  845   20]
 [   8    4    0    5   28    5    1   25   13  920]]


## Effects of Batch Size and Epochs

What will happen if we change the Epochs from 10 to 25

In [26]:
net = Simple_NN1(n_features=784)
train(net, train_images, train_labels, epochs=25, batch_size=100, device=device)
test(net, test_images, test_labels)

Device = cuda:0


  0%|          | 0/600 [00:00<?, ?it/s]C:\Users\igriffit\AppData\Local\Temp\ipykernel_25272\859838194.py:15: UserWarning: Implicit dimension choice for log_softmax has been deprecated. Change the call to include dim=X as an argument.
  x = F.log_softmax(self.fc4(x))
100%|██████████| 600/600 [00:01<00:00, 440.41it/s]


Iteration 0 | Loss: 2.25449800491333


100%|██████████| 600/600 [00:01<00:00, 493.04it/s]


Iteration 1 | Loss: 1.8916945457458496


...

Iteration 22 | Loss: 0.07687525451183319


100%|██████████| 600/600 [00:01<00:00, 443.73it/s]


Iteration 23 | Loss: 0.07437983900308609


100%|██████████| 600/600 [00:01<00:00, 457.09it/s]


Iteration 24 | Loss: 0.07217838615179062
Data Examples: tensor([-1.1617e+01, -1.2687e+01, -7.2707e+00, -4.9680e+00, -1.8502e+01,
        -1.1595e+01, -1.4392e+01, -7.9491e-03, -1.4601e+01, -8.3242e+00]) 

7
tensor([7, 2, 1, 0, 4])
tensor([7, 2, 1, 0, 4])
Accuracy: 0.955
[[ 963    0    1    2    0    7    1    4    2    0]
 [   0 1116    2    3    0    1    3    2    8    0]
 [   7    1  987    8    3    0    9    7    9    1]
 [   0    0    8  956    0   21    0   11   13    1]
 [   1    1    4    0  931    0    9    4    3   29]
 [   8    1    0    8    2  849    7    2    9    6]
 [  10    4    1    1    5   12  919    2    4    0]
 [   1   10   14    4    1    1    0  975    0   22]
 [   6    2    4   10    7    8    4    4  916   13]
 [   8    3    1    7   22    7    0    8   11  942]]


In [27]:
net = Simple_NN1(n_features=784)
train(net, train_images, train_labels, epochs=10, batch_size=1, device=device)
test(net, test_images, test_labels)

Device = cuda:0


  0%|          | 0/60000 [00:00<?, ?it/s]C:\Users\igriffit\AppData\Local\Temp\ipykernel_25272\859838194.py:15: UserWarning: Implicit dimension choice for log_softmax has been deprecated. Change the call to include dim=X as an argument.
  x = F.log_softmax(self.fc4(x))
 12%|█▏        | 7365/60000 [00:12<01:28, 593.48it/s]


KeyboardInterrupt: 

In [28]:
net = Simple_NN1(n_features=784)
train(net, train_images, train_labels, epochs=10, batch_size=30000, device=device)
test(net, test_images, test_labels)

Device = cuda:0


  0%|          | 0/2 [00:00<?, ?it/s]C:\Users\igriffit\AppData\Local\Temp\ipykernel_25272\859838194.py:15: UserWarning: Implicit dimension choice for log_softmax has been deprecated. Change the call to include dim=X as an argument.
  x = F.log_softmax(self.fc4(x))
100%|██████████| 2/2 [00:00<00:00, 13.74it/s]


Iteration 0 | Loss: 2.3163530826568604


100%|██████████| 2/2 [00:00<00:00, 14.00it/s]


Iteration 1 | Loss: 2.316133499145508


...

Iteration 7 | Loss: 2.3148231506347656


100%|██████████| 2/2 [00:00<00:00, 16.30it/s]


Iteration 8 | Loss: 2.314605951309204


100%|██████████| 2/2 [00:00<00:00, 15.81it/s]


Iteration 9 | Loss: 2.3143885135650635
Data Examples: tensor([-2.0856, -2.4407, -2.1477, -2.3506, -2.6130, -2.1019, -2.2756, -2.2240,
        -2.4613, -2.4671]) 

7
tensor([0, 0, 0, 0, 0])
tensor([7, 2, 1, 0, 4])
Accuracy: 0.099
[[ 980    0    0    0    0    0    0    0    0    0]
 [1125    0    0    0    0   10    0    0    0    0]
 [ 991    0    0    0    0   41    0    0    0    0]
 [ 994    0    0    0    0   16    0    0    0    0]
 [ 920    0    0    0    0   62    0    0    0    0]
 [ 884    0    0    0    0    8    0    0    0    0]
 [ 950    0    0    0    0    8    0    0    0    0]
 [ 894    0    0    0    0  134    0    0    0    0]
 [ 941    0    0    0    0   33    0    0    0    0]
 [ 892    0    0    0    0  117    0    0    0    0]]


In [ ]:
# Effects of Batch Size
# https://medium.com/mini-distill/effect-of-batch-size-on-training-dynamics-21c14f7a716e

# Tutorial Question 1:

Create a Neural Network with the following structure:

784 Neurons -> 128 Neurons -> 64 Neurons -> 10 Neurons

# Tutorial Question 2:

Create a Neural Network with the following structure:

784 Neurons -> 256 Neurons -> 128 Neurons -> 64 Neurons -> 32 Neurons -> 10 Neurons

# Tutorial Question 3:

Attempt using a different activation function for the hidden layers. Try googling a different activation function to use!

# Tutorial Question 4 (Additional):

Google for another Image Recognition Dataset, try loading the data into PyTorch as a PyTorch Tensor.
Note: You may need to research how to do this!